<a href="https://colab.research.google.com/github/louistrue/DB-1/blob/main/DB1_Projektarbeit_Toolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DB1 — Toolkit für die Projektarbeit

**Vorlagen, die du in deiner Projektarbeit direkt brauchst.** Jede Sektion ist eigenständig, steig direkt dort ein wo du Hilfe brauchst.

---

## Inhalt

1. **Daten lesen vor Code schreiben** — die vier Standardzeilen, die immer zuerst kommen
2. **Plausibilität konkret** — drei Pattern: Zahlen-Fenster, Stichprobe, Summen-Check
3. **Plotly-Schnellstart** — drei Diagramme aus einem Dictionary
4. **Mini-Hilfe** — Schnell-Tricks, wenn du nicht weiterkommst

---

## Vorbereitung

Dieses Notebook arbeitet mit zwei Beispiel-Datensätzen:

- **`bruecke_a.json`** — eine kleine Velobrücke (14 Stäbe, Stahl + Holz)
- **`staebe.csv`** — dieselben Stäbe als CSV-Tabelle

**So lädst du die Dateien in Colab hoch:**

1. Klicke oben links auf das **Ordner-Symbol** (Dateien-Panel)
2. Drücke auf **„Hochladen"** (Symbol mit Pfeil nach oben)
3. Wähle `bruecke_a.json` und `staebe.csv` aus
4. Die Dateien erscheinen im Panel und sind sofort verwendbar

Wenn du das Notebook neu startest, musst du die Dateien neu hochladen — Colab löscht sie zwischen den Sitzungen.


---

# 1. Daten lesen vor Code schreiben

**Goldene Regel: Lies zuerst, was in deinen Daten drin ist. Erst dann schreibst du Code.**

Wer mit unbekannten Daten Code schreibt, schreibt Code für Daten, die er sich vorstellt — nicht für die, die wirklich da sind. Das endet in Fehlern, die schwer zu finden sind.

## JSON-Datei laden


In [ ]:
import json

with open("bruecke_a.json", "r", encoding="utf-8") as f:
    daten = json.load(f)

print("Geladen:", type(daten))


## Die vier Standardzeilen

Diese vier Befehle gehen am Anfang **jeder** Auswertung. Sie beantworten: *Was habe ich überhaupt?*


In [ ]:
# Die vier Standardzeilen — IMMER zuerst:
print("1. Typ:        ", type(daten))           # Was ist das? dict / list / ...
print("2. Top-Level:  ", list(daten.keys()))    # Falls dict: welche Schlüssel?
print("3. Anzahl:     ", len(daten["staebe"]))  # Wie viele Elemente?
print("4. Erstes:     ", daten["staebe"][0])    # Wie sieht ein Element aus?


**Was du daraus liest:**

- Top-Level ist ein Dictionary mit `name`, `spannweite_m`, `anzahl_felder` und `staebe`
- Es gibt 14 Stäbe
- Pro Stab: `id`, `typ`, `material`, `laenge_m`, `querschnittsflaeche_m2`

Jetzt — und erst jetzt — weisst du, was du iterieren und auswerten kannst. Vor diesen vier Zeilen ist jeder Code blind.

## CSV-Datei laden

Für CSV-Daten (Excel-Export, Open-Data-Portale, eigene Messungen) ist `pandas` der Standard.


In [ ]:
import pandas as pd

df = pd.read_csv("staebe.csv")

print(df.head())               # Erste 5 Zeilen anschauen
print()
print("Spalten:", df.columns.tolist())
print("Form (Zeilen, Spalten):", df.shape)
print("Datentypen pro Spalte:")
print(df.dtypes)


## IFC-Datei laden

Für IFC-Modelle (BIM-Standard) brauchst du `ifcopenshell`. Das Modul ist in Colab nicht vorinstalliert — beim ersten Mal also installieren:


In [ ]:
# Einmalig in Colab ausführen:
# !pip install ifcopenshell

# Dann das Pattern (auskommentieren wenn du eine .ifc-Datei hast):
#
# import ifcopenshell
# import ifcopenshell.util.element
#
# modell = ifcopenshell.open("mein_modell.ifc")
#
# # Was steckt drin?
# print("Schema:", modell.schema)              # IFC2X3 / IFC4 / ...
# print("Total Entities:", len(modell.by_type("IfcRoot")))
#
# # Wie viele Stützen, Träger, Wände?
# print("Stützen:", len(modell.by_type("IfcColumn")))
# print("Träger:", len(modell.by_type("IfcBeam")))
# print("Wände:",  len(modell.by_type("IfcWall")))
#
# # Erstes Element anschauen
# erste_stuetze = modell.by_type("IfcColumn")[0]
# print("Name:", erste_stuetze.Name)
# print("Material:", ifcopenshell.util.element.get_material(erste_stuetze))

print("IFC-Pattern auskommentiert — verwende es wenn du eine IFC-Datei hochgeladen hast.")


## Häufige Stolperfallen beim Daten-Laden

- **`FileNotFoundError`**: Die Datei ist nicht im richtigen Ordner. Prüfe das Files-Panel (Ordner-Symbol oben links). Datei muss direkt im `/content`-Ordner liegen, oder kompletter Pfad angeben.
- **`JSONDecodeError`**: Datei ist kein gültiges JSON. Manchmal beginnt sie mit BOM oder hat ein Komma am Ende. Mit Texteditor öffnen und prüfen.
- **CSV mit Komma vs. Semikolon**: Schweizer Excel exportiert oft `;` statt `,`. Dann: `pd.read_csv("datei.csv", sep=";")`.
- **Encoding-Probleme bei Umlauten**: `pd.read_csv("datei.csv", encoding="latin-1")` oder `encoding="utf-8-sig"`.


---

# 2. Plausibilität konkret machen

**Eine berechnete Zahl ohne Plausibilitäts-Check ist gefährlich.** Code rechnet, was du sagst — nicht, was du meinst.

Aussagen wie *"sieht plausibel aus"* oder *"ist wahrscheinlich richtig"* zählen nicht. Eine Plausibilitäts-Probe hat einen **konkreten Zielwert** oder **eine Vergleichszahl**.

Drei Pattern, die fast immer funktionieren.

## Pattern A: Zahlen-Fenster aus Erfahrung

Wenn du Mengen, Massen, Längen, Volumen berechnest: Schreib vor der Auswertung auf, in welchem Bereich das Ergebnis liegen *muss*, damit es realistisch ist.

**Beispiel:**
- Velobrücke: 100-600 kg pro Meter Länge (typischer Bereich)
- Wenn meine Rechnung 20 kg/m oder 2000 kg/m ergibt → Fehler
- Wenn meine Rechnung 200 kg/m ergibt → plausibel


In [ ]:
# Massenberechnung der Brücke
dichte_kg_m3 = {"Stahl": 7850, "Holz": 460}

masse_total = 0
for stab in daten["staebe"]:
    A = stab["querschnittsflaeche_m2"]
    L = stab["laenge_m"]
    rho = dichte_kg_m3[stab["material"]]
    masse_total += A * L * rho

print(f"Gesamtmasse: {masse_total:.1f} kg")

# Plausibilitäts-Probe: Masse pro Meter Spannweite
spannweite = daten["spannweite_m"]
masse_pro_meter = masse_total / spannweite
print(f"Spannweite: {spannweite} m")
print(f"Masse pro Meter Spannweite: {masse_pro_meter:.1f} kg/m")

# Erwarteter Bereich für eine Velobrücke
unten, oben = 100, 600
if unten <= masse_pro_meter <= oben:
    print(f"OK — liegt im erwarteten Bereich {unten}-{oben} kg/m")
else:
    print(f"WARNUNG — liegt ausserhalb {unten}-{oben} kg/m. Fehler suchen!")


**Wie du den Zahlen-Bereich findest:**

- Erfahrungswerte aus Vorlesungen (z.B. 2.3 t/m² Asphaltdecke)
- Normen (SIA-Tabellen für Lasten, Verformungen)
- Faustregeln aus dem Praktikum
- Wikipedia / Datenblätter für Materialdichten
- KI fragen — und die Antwort mit zweiter Quelle prüfen

## Pattern B: Stichprobe

Wenn dein Datensatz zu gross ist, um Erfahrungswerte direkt anzuwenden, prüfe einen kleinen Ausschnitt manuell.

**Beispiel:**
- 63 000 Brücken in der Schweiz aus OpenStreetMap
- 5×5 Zufallsstichprobe in Google Maps gegenchecken → 25/25 wirklich Brücken
- Daher: die 63 000 sind realistisch


In [ ]:
import random

# Zufallsstichprobe aus den Stäben
random.seed(42)
stichprobe = random.sample(daten["staebe"], k=3)

print("Stichprobe (manuell prüfen):")
for stab in stichprobe:
    print(f"  {stab['id']:3s}  {stab['typ']:12s}  {stab['material']:6s}  L = {stab['laenge_m']} m  A = {stab['querschnittsflaeche_m2']} m²")

print()
print("Du gehst diese Zeilen manuell durch und schaust:")
print("- Sind die Materialien sinnvoll?")
print("- Sind die Längen realistisch?")
print("- Stimmen die Werte mit einer anderen Quelle überein?")


## Pattern C: Summen-Check

Eine dritte einfache Probe: prüfe Summen, die du eigentlich kennst.

- *Summe aller Anteile = 100 %*
- *Anzahl gezählter Bauteile = Anzahl Datensätze*
- *Σ Teilmassen = Gesamtmasse*

Wenn das nicht aufgeht, hast du irgendwo doppelt gezählt oder etwas vergessen.


In [ ]:
# Anteile pro Material müssen 100 % ergeben
anteile = {}
for stab in daten["staebe"]:
    m = stab["material"]
    anteile[m] = anteile.get(m, 0) + 1

total = sum(anteile.values())
print(f"Total Stäbe: {total}")
print()
print("Anzahl pro Material:")
summe_prozent = 0
for m, n in anteile.items():
    prozent = 100 * n / total
    summe_prozent += prozent
    print(f"  {m}: {n} Stäbe ({prozent:.1f} %)")

print()
print(f"Summe Anteile: {summe_prozent:.1f} %")
print("Wenn das nicht 100.0 ist, stimmt etwas nicht.")


---

# 3. Plotly-Schnellstart

Drei Diagramme aus einem Aggregations-Dictionary in wenigen Zeilen. Reicht für 80 % aller Projektarbeiten.

## Aggregation vorbereiten

Wir starten mit einem Dictionary `{Material: Wert}`. Genau das produzieren die Skelette aus der Vorlesung.


In [ ]:
# Längen pro Material
laenge_pro_material = {}
for stab in daten["staebe"]:
    m = stab["material"]
    laenge_pro_material[m] = laenge_pro_material.get(m, 0) + stab["laenge_m"]

print(laenge_pro_material)


## Bar-Chart — für absolute Vergleiche


In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(x=list(laenge_pro_material.keys()),
           y=list(laenge_pro_material.values()))
])
fig.update_layout(
    title="Länge pro Material",
    xaxis_title="Material",
    yaxis_title="Länge (m)",
)
fig.show()


## Pie-Chart — für Anteile am Ganzen


In [ ]:
fig = go.Figure(data=[
    go.Pie(labels=list(laenge_pro_material.keys()),
           values=list(laenge_pro_material.values()))
])
fig.update_layout(title="Anteil der Materialien")
fig.show()


## Scatter — für Zusammenhänge

Hier brauchst du zwei Listen: x-Werte und y-Werte.


In [ ]:
# Länge vs. Querschnittsfläche pro Stab
x_werte = [s["laenge_m"] for s in daten["staebe"]]
y_werte = [s["querschnittsflaeche_m2"] for s in daten["staebe"]]
labels  = [s["id"] for s in daten["staebe"]]

fig = go.Figure(data=[
    go.Scatter(x=x_werte, y=y_werte, mode="markers+text",
               text=labels, textposition="top center")
])
fig.update_layout(
    title="Länge vs. Querschnittsfläche pro Stab",
    xaxis_title="Länge (m)",
    yaxis_title="Querschnittsfläche (m²)",
)
fig.show()


## Welches Diagramm wann?

| Frage | Diagramm |
|---|---|
| Wer ist grösser/kleiner? (absolute Vergleiche) | **Bar-Chart** |
| Wie ist die Aufteilung? (Anteile, Prozent) | **Pie-Chart** |
| Hängen zwei Werte zusammen? | **Scatter** |
| Wie verändert sich was über die Zeit / einen Parameter? | **Line-Chart** (`mode="lines"`) |

**Faustregel:** Pie-Chart nur für Anteile bei einem Datensatz. Für alles andere lieber Bar.


---

# 4. Mini-Hilfe: wenn du nicht weiterkommst

Drei eingebaute Python-Werkzeuge, die dich schnell entstauen.

## `?` und `help()` — was kann diese Funktion?

In Colab schreibst du ein Fragezeichen hinter eine Funktion, um die Doku zu sehen.


In [ ]:
# In Colab/Jupyter mit ? — gibt Doku aus:
# json.load?
# pd.read_csv?

# Funktioniert immer:
help(len)


## `dir()` — welche Methoden hat dieses Objekt?

Wenn du nicht weisst, was du mit einem Objekt machen kannst:


In [ ]:
# Was kann ich mit einem String alles machen?
text = "Hallo Welt"
print([m for m in dir(text) if not m.startswith("_")])

# Beispiel: text.upper() macht alles gross
# text.replace("Welt", "BFH") ersetzt


## `type()` — was ist das für ein Ding?

Wenn du nicht weisst, ob du mit einer Liste, einem Dictionary oder einem DataFrame zu tun hast:


In [ ]:
print(type(daten))                    # dict
print(type(daten["staebe"]))          # list
print(type(daten["staebe"][0]))       # dict
print(type(daten["staebe"][0]["id"])) # str


## Häufige Fehler — Lookup-Tabelle

| Fehlermeldung | Bedeutung | Erste Hilfe |
|---|---|---|
| `KeyError: 'xyz'` | Schlüssel existiert nicht im Dictionary | `print(list(daten.keys()))` — gibts den Schlüssel wirklich? |
| `IndexError: list index out of range` | Du greifst auf ein Element zu, das nicht existiert | `len(meine_liste)` prüfen |
| `TypeError: ... must be str, not int` | Du mischst Datentypen | `type(x)` und `type(y)` prüfen, ggf. `str(x)` oder `int(x)` |
| `NameError: name 'xyz' is not defined` | Variable existiert nicht (Tippfehler oder noch nicht zugewiesen) | Vorherige Zellen ausgeführt? |
| `ValueError: could not convert string to float` | Du willst `float("abc")` machen | Daten enthalten Text statt Zahl |
| `FileNotFoundError` | Datei nicht da | Files-Panel prüfen, Pfad korrigieren |

## Colab-Tastaturkürzel (die wichtigsten)

- `Shift + Enter` — Zelle ausführen, weiter zur nächsten
- `Ctrl + Enter` — Zelle ausführen, bleiben
- `Ctrl + M, B` — neue Zelle unten
- `Ctrl + M, A` — neue Zelle oben
- `Ctrl + /` — auskommentieren / einkommentieren
- `Ctrl + S` — speichern (geht automatisch, aber gut zu wissen)

---

## Wenn nichts mehr hilft

1. Speichere das Notebook (`Ctrl + S`).
2. Probiere: *Laufzeit → Sitzung neu starten und alle Zellen ausführen*. Behebt 50 % aller Probleme.
3. Wenn das nicht hilft: Frag im Moodle-Forum, im 1:1, oder per Mail.


---

# Geschafft

Du hast jetzt:

- **Vier Standardzeilen** für jeden neuen Datensatz
- **Drei Plausibilitäts-Pattern** (Zahlen-Fenster, Stichprobe, Summen-Check)
- **Drei Plotly-Diagramme** in wenigen Zeilen
- **Mini-Hilfe** für die häufigsten Fehler

**Nächste Schritte in deinem Projekt:**

1. Datensatz öffnen, die vier Standardzeilen ausführen
2. Frage prüfen: passt sie zu dem, was im Datensatz drin ist?
3. Pseudo-Code oder Python schreiben
4. Plausibilitäts-Probe einbauen (Pattern A, B oder C)
5. Ergebnis visualisieren (mindestens ein Diagramm)
6. Reflexion schreiben (inkl. KI-Nutzung)

Viel Erfolg.
